# nbdev_tools

> MCP tools for running nbdev commands (prepare, export, test)

In [ ]:
#| default_exp tools.nbdev

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional

from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations

from nbdev_mcp.utils.paths import (
    project_summary,
    resolve_selector,
    nbdev_command_name,
    nbdev_generation,
)
from nbdev_mcp.utils.subprocess import run, wrap_with_env
from nbdev_mcp.utils.rich import render_result

## nbdev Commands

In [ ]:
#| export
def nbdev_prepare(
    project: Optional[str] = None,
    extra_args: Optional[List[str]] = None,
    use_env: bool = True
) -> Dict[str, Any]:
    """Run nbdev prepare for the project with version-aware command detection.
    
    Uses detected nbdev generation to choose command style:
    - v2: ``nbdev_prepare``
    - v3: ``nbdev-prepare``
    Falls back to the alternate form when the preferred command is missing.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    extra_args : List[str], optional
        Additional arguments to pass to the command.
    use_env : bool
        If True, run in project's conda/mamba environment.
    
    Returns
    -------
    Dict[str, Any]
        Result with command output, chosen command, and detection metadata.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}

    preferred = nbdev_command_name(p, "prepare")
    alternate = "nbdev-prepare" if preferred == "nbdev_prepare" else "nbdev_prepare"
    candidates = [preferred]
    if alternate not in candidates:
        candidates.append(alternate)

    logs: Dict[str, Any] = {}
    used_command = preferred
    for i, command_name in enumerate(candidates):
        cmd = wrap_with_env([command_name] + (extra_args or []), p, use_env)
        logs = run(cmd, cwd=p)
        used_command = command_name
        if logs.get("ok"):
            break

        stderr = (logs.get("stderr") or "").lower()
        stdout = (logs.get("stdout") or "").lower()
        command_missing = (
            "not found" in stderr
            or "no such file or directory" in stderr
            or "command not found" in stderr
            or "not recognized as an internal or external command" in stderr
            or "not found" in stdout
        )
        if i == len(candidates) - 1 or not command_missing:
            break

    pretty = render_result(used_command, project_summary(p), logs)
    return {
        **logs,
        'project': str(p),
        'nbdev_generation': nbdev_generation(p),
        'nbdev_command': used_command,
        'nbdev_command_candidates': candidates,
        'pretty': pretty,
    }


In [ ]:
#| export
def nbdev_export(
    project: Optional[str] = None,
    processors: Optional[List[str]] = None,
    extra_args: Optional[List[str]] = None,
    use_env: bool = True,
    ensure_ids: bool = True
) -> Dict[str, Any]:
    """Run nbdev export for the project with version-aware command detection.
    
    Uses detected nbdev generation to choose command style:
    - v2: ``nbdev_export``
    - v3: ``nbdev-export``
    Falls back to the alternate form when the preferred command is missing.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    processors : List[str], optional
        Pre/post processors to apply.
    extra_args : List[str], optional
        Additional arguments to pass to the command.
    use_env : bool
        If True, run in project's conda/mamba environment.
    
    Returns
    -------
    Dict[str, Any]
        Result with command output, chosen command, and detection metadata.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}

    ids_stamped: Dict[str, Any] = {}
    if ensure_ids:
        try:
            import glob, os
            from nbdev_mcp.utils.nbformat import stamp_notebook_ids
            nbs_root = os.path.join(str(p), 'nbs')
            if not os.path.isdir(nbs_root): nbs_root = str(p)
            for f in glob.glob(os.path.join(nbs_root, '**', '*.ipynb'), recursive=True):
                if '.ipynb_checkpoints' in f: continue
                n = stamp_notebook_ids(f)
                if n: ids_stamped[os.path.relpath(f, str(p))] = n
        except Exception as e:
            ids_stamped = {'_error': str(e)}

    command_args: List[str] = []
    if processors:
        command_args += ['--procs'] + processors
    if extra_args:
        command_args += extra_args

    preferred = nbdev_command_name(p, "export")
    alternate = "nbdev-export" if preferred == "nbdev_export" else "nbdev_export"
    candidates = [preferred]
    if alternate not in candidates:
        candidates.append(alternate)

    logs: Dict[str, Any] = {}
    used_command = preferred
    for i, command_name in enumerate(candidates):
        cmd = wrap_with_env([command_name] + command_args, p, use_env)
        logs = run(cmd, cwd=p)
        used_command = command_name
        if logs.get("ok"):
            break

        stderr = (logs.get("stderr") or "").lower()
        stdout = (logs.get("stdout") or "").lower()
        command_missing = (
            "not found" in stderr
            or "no such file or directory" in stderr
            or "command not found" in stderr
            or "not recognized as an internal or external command" in stderr
            or "not found" in stdout
        )
        if i == len(candidates) - 1 or not command_missing:
            break

    pretty = render_result(used_command, project_summary(p), logs)
    return {
        **logs,
        'project': str(p),
        'nbdev_generation': nbdev_generation(p),
        'nbdev_command': used_command,
        'nbdev_command_candidates': candidates,
        'ids_stamped': ids_stamped,
        'pretty': pretty,
    }


In [ ]:
#| export
def nbdev_test(
    project: Optional[str] = None,
    path: Optional[str] = None,
    flags: str = '',
    n_workers: Optional[int] = None,
    do_print: bool = False,
    file_re: Optional[str] = None,
    use_env: bool = True
) -> Dict[str, Any]:
    """Run nbdev test for the project with version-aware command detection.
    
    Uses detected nbdev generation to choose command style:
    - v2: ``nbdev_test``
    - v3: ``nbdev-test``
    Falls back to the alternate form when the preferred command is missing.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    path : str, optional
        Specific path to test.
    flags : str
        Test flags to pass.
    n_workers : int, optional
        Number of parallel workers.
    do_print : bool
        If True, print output during testing.
    file_re : str, optional
        Regex to filter test files.
    use_env : bool
        If True, run in project's conda/mamba environment.
    
    Returns
    -------
    Dict[str, Any]
        Result with command output, chosen command, and detection metadata.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}

    command_args: List[str] = []
    if path:
        command_args += ['--path', path]
    if flags:
        command_args += ['--flags', flags]
    if n_workers is not None:
        command_args += ['--n_workers', str(n_workers)]
    if do_print:
        command_args += ['--do_print']
    if file_re:
        command_args += ['--file_re', file_re]

    preferred = nbdev_command_name(p, "test")
    alternate = "nbdev-test" if preferred == "nbdev_test" else "nbdev_test"
    candidates = [preferred]
    if alternate not in candidates:
        candidates.append(alternate)

    logs: Dict[str, Any] = {}
    used_command = preferred
    for i, command_name in enumerate(candidates):
        cmd = wrap_with_env([command_name] + command_args, p, use_env)
        logs = run(cmd, cwd=p)
        used_command = command_name
        if logs.get("ok"):
            break

        stderr = (logs.get("stderr") or "").lower()
        stdout = (logs.get("stdout") or "").lower()
        command_missing = (
            "not found" in stderr
            or "no such file or directory" in stderr
            or "command not found" in stderr
            or "not recognized as an internal or external command" in stderr
            or "not found" in stdout
        )
        if i == len(candidates) - 1 or not command_missing:
            break

    pretty = render_result(used_command, project_summary(p), logs)
    return {
        **logs,
        'project': str(p),
        'nbdev_generation': nbdev_generation(p),
        'nbdev_command': used_command,
        'nbdev_command_candidates': candidates,
        'pretty': pretty,
    }


In [ ]:
#| export
def pytest_run(
    project: Optional[str] = None,
    args: Optional[List[str]] = None,
    use_env: bool = True
) -> Dict[str, Any]:
    """Run pytest on the project's tests/ directory.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    args : List[str], optional
        Arguments to pass to pytest (default ['-q']).
    use_env : bool
        If True, run in project's conda/mamba environment.
    
    Returns
    -------
    Dict[str, Any]
        Result with command output and status.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    cmd = wrap_with_env(['pytest'] + (args or ['-q']), p, use_env)
    logs = run(cmd, cwd=p)
    pretty = render_result('pytest', project_summary(p), logs)
    return {**logs, 'project': str(p), 'pretty': pretty}

## MCP Registration

In [ ]:
#| export
# Tool annotation definitions for nbdev build tools
NBDEV_TOOL_ANNOTATIONS = {
    'nbdev_prepare': ToolAnnotations(
        title="NBDev Prepare",
        readOnlyHint=False,
        idempotentHint=False,
        openWorldHint=False
    ),
    'nbdev_export': ToolAnnotations(
        title="NBDev Export",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'nbdev_test': ToolAnnotations(
        title="NBDev Test",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'pytest_run': ToolAnnotations(
        title="Pytest Run",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
}

def add_nbdev_tools(mcp: FastMCP) -> None:
    """Attach nbdev build/test tools to the MCP server with annotations.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    mcp.add_tool(nbdev_prepare, annotations=NBDEV_TOOL_ANNOTATIONS['nbdev_prepare'])
    mcp.add_tool(nbdev_export, annotations=NBDEV_TOOL_ANNOTATIONS['nbdev_export'])
    mcp.add_tool(nbdev_test, annotations=NBDEV_TOOL_ANNOTATIONS['nbdev_test'])
    mcp.add_tool(pytest_run, annotations=NBDEV_TOOL_ANNOTATIONS['pytest_run'])

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()